# Quick Test for EPANG-Gen Optimizers
This notebook tests EPANG-Gen and EPANG-Gen-light on a simple problem.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from epang_gen.optimizers import EPANGGen, ManualADOPT
from epang_gen.models import BayesianPINN, BayesianPASA
from epang_gen.problems import poisson_1d_loss, generate_poisson_1d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Test function
def test_epang_variants():
    torch.manual_seed(42)
    
    # Create models
    model_full = BayesianPINN([1, 20, 20, 1]).to(device)
    model_light = BayesianPINN([1, 20, 20, 1]).to(device)
    model_light.load_state_dict(model_full.state_dict())
    
    # Optimizers
    pasa = BayesianPASA(initial_rank=5)
    opt_full = EPANGGen(model_full.parameters(), lr=0.01, rank=5, pasa=pasa)
    opt_light = EPANGGen(model_light.parameters(), lr=0.01, rank=5,
                         eigen_update_freq=10000, use_curvature_lr=False)
    
    # Data
    x_colloc, x_bc, u_bc = generate_poisson_1d(200, 2)
    x_colloc = x_colloc.to(device)
    x_bc = x_bc.to(device)
    u_bc = u_bc.to(device)
    
    # Training
    loss_full, loss_light = [], []
    epochs = 300
    
    for epoch in range(epochs):
        # Full EPANG
        def closure_full():
            opt_full.zero_grad()
            loss = poisson_1d_loss(model_full, x_colloc, x_bc, u_bc)
            loss.backward()
            return loss.item()
        l_full = opt_full.step(closure_full)
        loss_full.append(l_full)
        
        # Light EPANG
        def closure_light():
            opt_light.zero_grad()
            loss = poisson_1d_loss(model_light, x_colloc, x_bc, u_bc)
            loss.backward()
            return loss.item()
        l_light = opt_light.step(closure_light)
        loss_light.append(l_light)
    
    print(f"EPANG-Full final loss: {loss_full[-1]:.6f}")
    print(f"EPANG-Light final loss: {loss_light[-1]:.6f}")
    print(f"Light/Full ratio: {loss_light[-1]/loss_full[-1]:.3f}")
    
    plt.figure(figsize=(10,4))
    plt.plot(loss_full, label='EPANG-Full')
    plt.plot(loss_light, label='EPANG-Light')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return loss_full, loss_light

test_epang_variants()